# 02 - First PINN: ODE

Solve $du/dt = -0.5u$, $u(0)=1$

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, sys
sys.path.insert(0, '../src')
from pinns import MLP, set_seed
from pinns.utils.derivatives import gradient
set_seed(42)
lam = 0.5
model = MLP(input_dim=1, output_dim=1, hidden_dims=[32,32,32], activation='tanh')

In [ ]:
def pde_loss(model, t):
    t = t.requires_grad_(True)
    u = model(t)
    du = gradient(u, t)
    return torch.mean((du + lam * u) ** 2)
def ic_loss(model):
    return torch.mean((model(torch.zeros(10, 1)) - 1.0) ** 2)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(1500):
    opt.zero_grad()
    loss = pde_loss(model, torch.rand(50, 1) * 3) + 10 * ic_loss(model)
    loss.backward(); opt.step()
    if (epoch+1) % 500 == 0: print(f'Epoch {epoch+1}: {loss.item():.4e}')

In [ ]:
t = torch.linspace(0, 3, 100).reshape(-1, 1)
with torch.no_grad(): u_pred = model(t)
u_exact = torch.exp(-lam * t)
plt.figure(figsize=(10,4))
plt.plot(t.numpy(), u_exact.numpy(), 'b-', lw=2, label='Exact')
plt.plot(t.numpy(), u_pred.numpy(), 'r--', lw=2, label='PINN')
plt.legend(); plt.grid(True, alpha=0.3)
plt.xlabel('t'); plt.ylabel('u(t)'); plt.title('Exponential Decay')
plt.tight_layout(); plt.show()
print(f'MAE: {torch.abs(u_pred - u_exact).mean():.6e}')